In [1]:
import pandas as pd
import glob

# Unificar los ficheros de datos

In [2]:
archivos = glob.glob("Datos/Originales/Dowjones/*.csv")
dfs = [pd.read_csv(f) for f in archivos]
df_unificado = pd.concat(dfs, ignore_index=True)
df_unificado

,open,high,low,last,close,volume,date,symbol,exchange
0,213.370,213.41,213.300,NaN,213.300,NaN,2018-08-16T21:30:00+0000,AAPL,IEXG
1,213.280,213.41,213.160,NaN,213.300,NaN,2018-08-16T21:00:00+0000,AAPL,IEXG
2,213.430,213.43,212.985,NaN,213.280,NaN,2018-08-16T20:30:00+0000,AAPL,IEXG
3,213.133,213.73,212.985,NaN,213.480,NaN,2018-08-16T20:00:00+0000,AAPL,IEXG
4,213.273,213.30,213.133,NaN,213.133,NaN,2018-08-16T19:30:00+0000,AAPL,IEXG
...,...,...,...,...,...,...,...,...,...
2744413,234.500,235.60,234.170,235.300,234.090,291094.0,2021-06-29T17:07:00+0000,V,IEXG
2744414,234.500,235.60,234.170,235.235,234.090,290571.0,2021-06-29T17:05:00+0000,V,IEXG
2744415,234.500,235.60,234.170,235.270,234.090,290254.0,2021-06-29T17:04:00+0000,V,IEXG
2744416,234.500,235.60,234.170,235.255,234.090,289854.0,2021-06-29T17:03:00+0000,V,IEXG


In [3]:
print(df_unificado.shape)
print(df_unificado["symbol"].nunique(), "acciones distintas")
print(df_unificado.dtypes)
print(df_unificado.head())

(2744418, 9)


26 acciones distintas
open        float64
high        float64
low         float64
last        float64
close       float64
volume      float64
date            str
symbol          str
exchange        str
dtype: object
      open    high      low  last    close  volume                      date  \
0  213.370  213.41  213.300   NaN  213.300     NaN  2018-08-16T21:30:00+0000   
1  213.280  213.41  213.160   NaN  213.300     NaN  2018-08-16T21:00:00+0000   
2  213.430  213.43  212.985   NaN  213.280     NaN  2018-08-16T20:30:00+0000   
3  213.133  213.73  212.985   NaN  213.480     NaN  2018-08-16T20:00:00+0000   
4  213.273  213.30  213.133   NaN  213.133     NaN  2018-08-16T19:30:00+0000   

  symbol exchange  
0   AAPL     IEXG  
1   AAPL     IEXG  
2   AAPL     IEXG  
3   AAPL     IEXG  
4   AAPL     IEXG  


# Limpiar el dataset

In [4]:
# 1. Nulos por columna (total y %)
print(df_unificado.isnull().sum())
print(df_unificado.isnull().mean() * 100)

open             0
high             0
low              0
last        849956
close       611651
volume      847408
date             0
symbol           0
exchange         0
dtype: int64


open         0.000000
high         0.000000
low          0.000000
last        30.970355
close       22.287093
volume      30.877512
date         0.000000
symbol       0.000000
exchange     0.000000
dtype: float64


- last (31% nulos): last es último valor recogido de la acción. Para este reto se usa close como precio de referencia, así que probablemente se pueda descartar last directamente en vez de imputarla. No aporta nada que no tengamos ya en close.
- close (22% nulos): esta si se necesita, porque de aqui se sacan los retornos logarítmicos y la volatilidad. 
- volume (31% nulos): para el reto, el volumen no es estrictamente necesario para la simulación Monte Carlo ni para el payoff, así que probablemente sea una columna secundaria (útil solo para el análisis descriptivo). 

In [5]:
# 2. Duplicados
print("Duplicados exactos:", df_unificado.duplicated().sum())
print("Duplicados por symbol+date:", df_unificado.duplicated(subset=["symbol", "date"]).sum())

Duplicados exactos: 0


Duplicados por symbol+date: 0


In [6]:
# 3. Rango de fechas y nº de filas por acción (para ver si todas cubren el mismo periodo)
df_unificado["date"] = pd.to_datetime(df_unificado["date"])
resumen = df_unificado.groupby("symbol")["date"].agg(["min", "max", "count"])
print(resumen)

                             min                       max   count
symbol                                                            
AAPL   2018-06-06 17:30:00+00:00 2021-07-01 00:00:00+00:00   48723
AMGN   2020-05-29 15:35:00+00:00 2021-07-01 00:00:00+00:00   25296
AXP    2020-05-29 15:30:59+00:00 2021-07-01 00:00:00+00:00   95650
BA     2020-05-29 15:35:51+00:00 2021-07-01 00:00:00+00:00  101436
CAT    2018-06-11 17:30:00+00:00 2021-07-01 00:00:00+00:00  122364
CRM    2020-05-29 15:31:30+00:00 2021-07-01 00:00:00+00:00  108823
CSCO   2020-05-29 15:31:26+00:00 2021-07-01 00:00:00+00:00  108585
CVX    2020-05-29 15:31:23+00:00 2021-07-01 00:00:00+00:00  108876
DIS    2020-05-29 15:35:00+00:00 2021-07-01 00:00:00+00:00  109393
DOW    2020-05-29 15:35:56+00:00 2021-07-01 00:00:00+00:00  106437
GS     2018-06-11 17:30:00+00:00 2021-07-01 00:00:00+00:00  133145
HD     2020-05-29 17:55:51+00:00 2021-07-01 00:00:00+00:00  105219
IBM    2020-05-29 17:56:29+00:00 2021-07-01 00:00:00+00:00  10

- 6 acciones (AAPL, CAT, GS, INTC, NKE, V) empiezan en junio 2018
- El resto (20 acciones) empiezan en mayo/junio 2020
- Todas terminan el 1 de julio de 2021

In [7]:
# 4. Valores de exchange (¿todas las acciones vienen de la misma bolsa/fuente?)
print(df_unificado["exchange"].value_counts())

exchange
IEXG    2744418
Name: count, dtype: int64


In [8]:
# Calculamos la diferencia de tiempo entre filas consecutivas, por cada símbolo
df_unificado = df_unificado.sort_values(["symbol", "date"])
df_unificado["diff_tiempo"] = df_unificado.groupby("symbol")["date"].diff()

# ¿Qué diferencias de tiempo hay y con qué frecuencia aparecen?
print(df_unificado["diff_tiempo"].value_counts().head(15))

diff_tiempo
0 days 00:01:00    2128282
0 days 00:02:00     363519
0 days 00:05:00     113350
0 days 00:03:00      28256
0 days 00:10:00      24415
0 days 00:15:00      20581
0 days 00:30:00      12084
0 days 00:04:00       8282
0 days 17:30:00       2649
0 days 04:01:00       1885
0 days 13:30:00       1723
0 days 00:06:00       1709
0 days 00:07:00       1065
0 days 05:02:00       1059
0 days 00:00:59        737
Name: count, dtype: int64


Tenemos datos intradía a granularidad de minuto, con algunos huecos naturales (fin de semana, cierre de sesión, minutos sin operaciones). No es un dataset diario, es mucho más fino de lo que probablemente se necesite para el reto.  

In [9]:
# 5. Validación de precios: ¿hay casos donde high < low, o close fuera de [low, high]?
inconsistencias = df_unificado[
    (df_unificado["high"] < df_unificado["low"]) |
    (df_unificado["close"] > df_unificado["high"]) |
    (df_unificado["close"] < df_unificado["low"])
]
print("Filas con precios inconsistentes:", len(inconsistencias))

Filas con precios inconsistentes: 761432


In [10]:
# 6. Valores negativos o cero donde no deberían
print("Precios <= 0:", (df_unificado[["open","high","low","close"]] <= 0).sum().sum())
print("Volumen negativo:", (df_unificado["volume"] < 0).sum())

Precios <= 0: 0
Volumen negativo: 0


# Preguntas

1. Enriquecimiento con Yahoo Finance -- > ¿Es para traer las 4 acciones que faltan del Dow Jones, para ampliar el histórico más allá de julio 2021, o para sustituir directamente el intradía por datos diarios limpios de Yahoo Finance? (sacar un dato por dia en vez de tener datos cada 30  mins por ejemplo)

2. Nuestros datos son intradía, con frecuencia que varía entre 1, 5 y 30 minutos según el año y la acción. Para la simulación Monte Carlo (5 años, 20 pasos) parece que necesitamos agregar a datos diarios, que hacemos un dato por dia? o trabajar con frecuencia de 30 minutos...

3. La columna last tiene un 31% de nulos y no la vemos necesaria si usamos close como precio de referencia. Se puede eliminar?

4. 6 de las 26 acciones tienen histórico desde 2018 y las otras 20 solo desde 2020. Si dos de nuestras 3 acciones de la cartera están en grupos distintos, ¿recortamos todas al periodo común (mayo 2020 - julio 2021), o es preferible elegir las 3 dentro del mismo grupo de cobertura?"

5. Un 27,7% de las filas tienen el cierre fuera del rango high/low de su barra. Tiene sentido o deberíamos limpiar/filtrar estas filas antes de agregar?

6. La columna volume tiene un 31% de nulos. No es imprescindible para Monte Carlo/payoff, pero si para el análisis descriptivo. La imputamos (ej. con 0, o interpolando) o la dejamos con nulos y simplemente los excluimos de los gráficos que la usen?"

# Respuestas a las preguntas: implementación

Resumen de las decisiones tomadas para las 6 preguntas de arriba (la justificación completa de cada una está debajo, en su sección):

1. **Frecuencia** → diaria. TTM=5 años y 20 pasos ⇒ dt = 5/20 = 0.25 años/paso; el Montecarlo solo consume `close`, así que trabajar a intradía no aporta nada y solo arrastra ruido y las inconsistencias de high/low.
2. **`last`** → se descarta (ya decidido arriba); no vuelve a aparecer.
3. **Cobertura desigual 2018 vs 2020** → en vez de truncar todo al periodo común, se amplía con Yahoo Finance una ventana común más larga para todas las acciones.
4. **Enriquecimiento con yfinance** → se usa para las 4 acciones que faltan del Dow Jones y para homogeneizar el histórico del resto, no solo para completar huecos puntuales.
5. **`close` fuera de [low, high]** → se repara expandiendo el rango (`high = max(high, close)`, `low = min(low, close)`) en vez de eliminar filas: no afecta a la simulación (solo usa `close`) y mejora la calidad de los gráficos descriptivos.
6. **`volume`** → se mantiene con nulos, sin imputar (decisión ya tomada arriba).

## 1. Agregar a frecuencia diaria

Los datos son intradía a distinta granularidad según acción y año (1, 5, 30 min...). Para el Montecarlo del reto (TTM=5 años, 20 pasos) el paso de simulación es dt=5/20=0.25 años, y la fórmula del payoff/GBM solo necesita `close`. No hay ninguna ganancia en calibrar con intradía y sí un coste: arrastramos el ruido de granularidad mixta y las inconsistencias de high/low que vimos arriba (27.7% de las filas).

Agregamos por (symbol, día): `open` el primero del día, `high` el máximo, `low` el mínimo, `close` el último, `volume` la suma (conservando NaN si todo el día estaba sin informar, coherente con la decisión de no imputar volumen con 0).

In [11]:
df_diario = (
    df_unificado
    .assign(fecha=df_unificado["date"].dt.date)
    .sort_values(["symbol", "date"])
    .groupby(["symbol", "fecha"], as_index=False)
    .agg(
        open=("open", "first"),
        high=("high", "max"),
        low=("low", "min"),
        close=("close", "last"),
        volume=("volume", lambda s: s.sum(min_count=1)),
        exchange=("exchange", "first"),
    )
)
df_diario = df_diario.rename(columns={"fecha": "date"})
df_diario["date"] = pd.to_datetime(df_diario["date"])
print(df_diario.shape)
df_diario.head()

(12301, 8)


,symbol,date,open,high,low,close,volume,exchange
0,AAPL,2018-06-06,193.630,194.00,191.950,194.00,NaN,IEXG
1,AAPL,2018-06-07,194.000,194.16,192.385,193.50,NaN,IEXG
2,AAPL,2018-06-08,193.500,193.50,189.805,191.51,NaN,IEXG
3,AAPL,2018-06-09,191.510,191.51,191.510,191.51,NaN,IEXG
4,AAPL,2018-06-11,191.345,191.97,190.230,191.16,NaN,IEXG


In [12]:
# Repetimos el chequeo de consistencia de precios sobre el dataset diario
inconsistencias_diario = df_diario[
    (df_diario["high"] < df_diario["low"]) |
    (df_diario["close"] > df_diario["high"]) |
    (df_diario["close"] < df_diario["low"])
]
n_incons_diario = len(inconsistencias_diario)
print(f"Filas con precios inconsistentes (diario): {n_incons_diario} de {len(df_diario)} "
      f"({n_incons_diario / len(df_diario) * 100:.2f}%)")

Filas con precios inconsistentes (diario): 1142 de 12301 (9.28%)


Agregar a diario baja las inconsistencias del 27.70% (intradía) al 9.28% (diario) — mejora mucho, pero no las elimina: no es solo un artefacto de la granularidad intradía, así que hace falta repararlas explícitamente.

## 2. Reparar `close` fuera de [low, high]

Por definición, `high`/`low` son los extremos de la barra y `close` es un precio dentro de ella, así que `low <= close <= high` debería cumplirse siempre. Cuando falla, el error lógico está en high/low (no en close), así que reparamos expandiendo el rango en vez de eliminar la fila — evita tirar 9-27% de los datos por un problema que solo toca 2 de 9 columnas.

**No afecta al resultado de la simulación**: el GBM/Montecarlo y el payoff solo usan `close`. Esto es una reparación de calidad para el análisis descriptivo y los gráficos (velas, etc.), no para el pipeline de valoración.

In [13]:
df_diario["high"] = df_diario[["high", "close"]].max(axis=1)
df_diario["low"] = df_diario[["low", "close"]].min(axis=1)

inconsistencias_tras_fix = df_diario[
    (df_diario["high"] < df_diario["low"]) |
    (df_diario["close"] > df_diario["high"]) |
    (df_diario["close"] < df_diario["low"])
]
print("Inconsistencias tras el arreglo:", len(inconsistencias_tras_fix))

Inconsistencias tras el arreglo: 0


## 3. Enriquecimiento con Yahoo Finance

Tenemos 26 de las 30 acciones del Dow Jones, y además una cobertura desigual: 6 acciones con histórico desde 2018 y 20 solo desde 2020. Para el Cholesky de la simulación hace falta un histórico **simultáneo** entre las 3 acciones elegidas — no se puede "rellenar" 2018-2020 para una acción que no tiene esos datos, es ausencia estructural, no un hueco.

En vez de truncar todas al periodo común (mayo 2020 - jul 2021, ~13 meses, poco para estimar una matriz de correlación estable), ampliamos con Yahoo Finance: trae histórico diario completo para cualquier ticker vigente, así que sirve tanto para las 4 acciones que faltan como para alargar el histórico de las 26 que ya tenemos. Elegimos una ventana de 5 años (2016-07-01 a 2021-07-01) porque coincide con el TTM del producto que se está valorando.

In [14]:
# Composición del Dow Jones 30 en el periodo del dataset (2018-2021)
dow30 = [
    "AAPL", "AMGN", "AXP", "BA", "CAT", "CRM", "CSCO", "CVX", "DIS", "DOW",
    "GS", "HD", "HON", "IBM", "INTC", "JNJ", "JPM", "KO", "MCD", "MMM",
    "MRK", "MSFT", "NKE", "PG", "TRV", "UNH", "V", "VZ", "WBA", "WMT",
]

simbolos_actuales = set(df_diario["symbol"].unique())
simbolos_faltantes = sorted(set(dow30) - simbolos_actuales)
print(f"{len(simbolos_actuales)} de {len(dow30)} acciones del Dow Jones ya en el dataset")
print("Faltan:", simbolos_faltantes)

26 de 30 acciones del Dow Jones ya en el dataset
Faltan: ['HON', 'VZ', 'WBA', 'WMT']


Descargamos de Yahoo Finance el histórico diario **sin ajustar** (`auto_adjust=False`) para las 30 acciones del Dow Jones, en la ventana 2016-07-01 a 2021-07-01. Lo hacemos ticker a ticker con manejo de errores porque no todas las acciones del Dow Jones de 2018-2021 siguen cotizando igual hoy (una acción puede haberse fusionado, cambiado de ticker o dejado de cotizar desde entonces).

In [15]:
import yfinance as yf

FECHA_INICIO = "2016-07-01"
FECHA_FIN = "2021-07-01"

partes = []
fallidos = []
for ticker in dow30:
    try:
        hist = yf.Ticker(ticker).history(start=FECHA_INICIO, end=FECHA_FIN, auto_adjust=False)
        if hist.empty:
            fallidos.append(ticker)
            continue
        hist = hist.reset_index()[["Date", "Open", "High", "Low", "Close", "Volume"]]
        hist.columns = ["date", "open", "high", "low", "close", "volume"]
        hist["date"] = pd.to_datetime(hist["date"]).dt.tz_localize(None)
        hist["symbol"] = ticker
        hist["exchange"] = "YFINANCE"
        partes.append(hist)
    except Exception as error:
        fallidos.append(ticker)
        print(f"{ticker}: fallo la descarga -> {error}")

df_yahoo = pd.concat(partes, ignore_index=True)
print("Descargadas correctamente:", df_yahoo["symbol"].nunique(), "de", len(dow30))
print("Fallidas:", fallidos)
print(df_yahoo.groupby("symbol")["date"].agg(["min", "max", "count"]))

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}


$WBA: possibly delisted; no timezone found


Descargadas correctamente: 29 de 30
Fallidas: ['WBA']
              min        max  count
symbol                             
AAPL   2016-07-01 2021-06-30   1258
AMGN   2016-07-01 2021-06-30   1258
AXP    2016-07-01 2021-06-30   1258
BA     2016-07-01 2021-06-30   1258
CAT    2016-07-01 2021-06-30   1258
CRM    2016-07-01 2021-06-30   1258
CSCO   2016-07-01 2021-06-30   1258
CVX    2016-07-01 2021-06-30   1258
DIS    2016-07-01 2021-06-30   1258
DOW    2019-03-20 2021-06-30    576
GS     2016-07-01 2021-06-30   1258
HD     2016-07-01 2021-06-30   1258
HON    2016-07-01 2021-06-30   1258
IBM    2016-07-01 2021-06-30   1258
INTC   2016-07-01 2021-06-30   1258
JNJ    2016-07-01 2021-06-30   1258
JPM    2016-07-01 2021-06-30   1258
KO     2016-07-01 2021-06-30   1258
MCD    2016-07-01 2021-06-30   1258
MMM    2016-07-01 2021-06-30   1258
MRK    2016-07-01 2021-06-30   1258
MSFT   2016-07-01 2021-06-30   1258
NKE    2016-07-01 2021-06-30   1258
PG     2016-07-01 2021-06-30   1258
TRV    201

**`WBA` falla siempre**: no es un cambio de ticker, es que la empresa ya no cotiza. Sycamore Partners cerró la adquisición de Walgreens Boots Alliance el 28 de agosto de 2025 (aprobada por el 96% de los accionistas el 11 de julio de 2025); desde entonces WBA opera como empresa privada y su acción dejó de negociarse en Nasdaq, por eso Yahoo Finance ya no sirve histórico bajo ningún símbolo para ella (comprobado buscando la empresa directamente en Yahoo Finance: 0 resultados). La dejamos fuera de la selección: de las 4 acciones que faltaban en el CSV original, solo recuperamos 3 (`HON`, `VZ`, `WMT`).

Fuentes: [Sycamore Partners Completes Acquisition of Walgreens Boots Alliance (Nasdaq)](https://www.nasdaq.com/press-release/sycamore-partners-completes-acquisition-walgreens-boots-alliance-2025-08-28), [Sycamore Partners Completes Acquisition of Walgreens Boots Alliance (Business Wire)](https://www.businesswire.com/news/home/20250827891313/en/Sycamore-Partners-Completes-Acquisition-of-Walgreens-Boots-Alliance)

**`DOW` solo tiene datos desde 2019-03-20**, incluso en Yahoo Finance: Dow Inc. se escindió de DowDuPont en abril de 2019 y no existía como acción cotizada antes de esa fecha. Es una limitación estructural, no de la fuente de datos — ningún proveedor va a tener más histórico. Por eso proponemos añadir **cobertura histórica suficiente** como criterio cuantitativo más de selección (paso 2 del reto) y descartar `DOW` salvo que haya un motivo cualitativo de peso para incluirla: con solo ~27 meses de histórico se queda muy por debajo de las ~28 acciones restantes (con datos completos desde 2016), y limitaría la ventana de calibración de σ/ρ para toda la cartera, no solo para ella.

### Por qué no mezclamos el CSV original y Yahoo Finance en la misma serie

Antes de usar Yahoo Finance como fuente de calibración, comprobamos que sea consistente con el CSV donde ambos se solapan. Lo hacemos con `AAPL`, comparando el `close` diario del CSV (agregado arriba) contra el `Close` de Yahoo para las mismas fechas.

In [16]:
comparacion_aapl = pd.merge(
    df_diario.loc[df_diario["symbol"] == "AAPL", ["date", "close"]].rename(columns={"close": "close_csv"}),
    df_yahoo.loc[df_yahoo["symbol"] == "AAPL", ["date", "close"]].rename(columns={"close": "close_yahoo"}),
    on="date", how="inner",
)
comparacion_aapl["diff_pct"] = (
    (comparacion_aapl["close_csv"] - comparacion_aapl["close_yahoo"]).abs()
    / comparacion_aapl["close_yahoo"] * 100
)

antes_split = comparacion_aapl[comparacion_aapl["date"] < "2020-08-31"]
despues_split = comparacion_aapl[comparacion_aapl["date"] >= "2020-08-31"]
print(f"Diferencia media ANTES del 2020-08-31: {antes_split['diff_pct'].mean():.2f}% (n={len(antes_split)})")
print(f"Diferencia media DESDE el 2020-08-31: {despues_split['diff_pct'].mean():.2f}% (n={len(despues_split)})")

Diferencia media ANTES del 2020-08-31: 299.63% (n=563)
Diferencia media DESDE el 2020-08-31: 2.93% (n=210)


La diferencia media pasa de ~300% antes del 31/08/2020 a ~2.9% después — justo la fecha del split de AAPL 4:1 (una acción pasa a ser cuatro, precio ÷4). Eso es exactamente lo que se ve: antes del split el CSV cotiza ~500$ y Yahoo ya lo da en términos post-split (~125$), una proporción ~4x consistente con el propio split, no ruido aleatorio.

**Qué hace `auto_adjust` y por qué no evita esto**: `auto_adjust=True` en yfinance recalcularía todo el histórico (open/high/low/close) para que sea comparable hoy, corrigiendo tanto splits como dividendos — es el modo pensado para retornos totales. Con `auto_adjust=False` se pide expresamente lo contrario: los precios "tal cual se negociaron" en cada fecha, con el ajuste de dividendos aparte en `Adj Close`. Por eso pedimos `False`: queríamos el precio histórico real, comparable al `close` del CSV. Pero empíricamente (esta comprobación) el `Close` que devuelve Yahoo Finance ya viene con los splits aplicados incluso con `auto_adjust=False` — solo los dividendos quedan fuera hasta pasar a `Adj Close`. Es un comportamiento del backend de datos de Yahoo, no algo documentado de forma fiable en yfinance, así que lo verificamos con datos reales en vez de fiarnos de la documentación.

**Decisión**: Yahoo Finance como fuente única para calibrar σ y ρ de las acciones elegidas — su ajuste por splits es consistente en todo el histórico, y mezclarlo con el CSV (sin ajustar) metería saltos artificiales como este en cualquier acción con splits en el periodo.

**¿Para qué sirve entonces el CSV, si no calibra nada?** Tres cosas, no es trabajo tirado:
1. Es el propio ejercicio de limpieza/EDA que pide el paso 1 del reto y que evalúa Data Science ("procesamiento justificado de las series") — se evalúa el proceso, no solo el resultado numérico final.
2. Es la comprobación de consistencia que acabamos de hacer: sin el CSV no habríamos detectado el problema del split, que sí podría haber contaminado σ si hubiéramos mezclado fuentes a ciegas.
3. Queda como fuente alternativa/de respaldo si Yahoo Finance no estuviera disponible en algún momento (rate limits, caída del servicio) durante el desarrollo del reto.

Guardamos ambos datasets en `Datos/Transformados/` (no se sube al repositorio, según la regla del curso): el CSV limpiado y agregado a diario, y el enriquecido de Yahoo Finance que alimentará la selección de las 3 acciones y la calibración de la simulación.

In [17]:
df_diario.to_csv("Datos/Transformados/dowjones_csv_diario_limpio.csv", index=False)
df_yahoo.to_csv("Datos/Transformados/dowjones_yahoo_diario.csv", index=False)
print("Guardado:", df_diario.shape, df_yahoo.shape)

Guardado: (12301, 8) (35800, 8)
